# Phase 2 — Neural Network Preprocessing (Full Samanantar Dataset)
**Merged from:** `Samanantar_Dataset_load.ipynb` + `Preprocessing.ipynb`

### Key changes from Phase 1:
- Uses the **entire Samanantar en-hi dataset (~10M pairs)** instead of 150K sample
- Data is loaded and filtered in **streaming chunks** to avoid RAM exhaustion
- Vocabulary built with `min_freq=10` (raised from 5) to keep vocab manageable at 10M scale
- `MAX_LEN` raised from 50 → 60 to capture more of the longer sentences in full data
- Tokenized data saved as **chunked CSV files** (not one giant pickle) for memory safety
- Embedding matrices built and saved as before (FastText 300-dim)
- Splits are 80/10/10 of the filtered data (same ratio as Phase 1)

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import re
import gc
import math
import pickle
import numpy as np
import pandas as pd
import nltk
from collections import Counter
from tqdm.auto import tqdm

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# CHANGE: output directory set to current working directory.
# On the university GPU server, change OUTPUT_DIR to your preferred path,
# e.g. '/home/username/mt_phase2/' or wherever you have disk space.
OUTPUT_DIR = './phase2_data/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_DIR}")

In [ ]:
# ── Cell 2: Load the full Samanantar en-hi dataset ───────────────────────────
# CHANGE: Phase 1 sampled 150K. Phase 2 uses the entire dataset.
# The dataset is ~10M pairs. `load_dataset` streams lazily from HuggingFace;
# set cache_dir to a disk location with enough space (the full dataset is ~3-4 GB).
from datasets import load_dataset

# CHANGE: Added cache_dir — point this to a directory with at least 5 GB free.
# On the university GPU this is usually /scratch/ or /data/.
CACHE_DIR = './hf_cache/'
os.makedirs(CACHE_DIR, exist_ok=True)

print("Loading Samanantar en-hi dataset (full)...")
print("This will download ~3-4 GB on first run — subsequent runs use the cache.")
dataset = load_dataset(
    "ai4bharat/samanantar",
    "hi",
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)
print(dataset)
print("\nSample pair:")
print(dataset['train'][0])

In [ ]:
# ── Cell 3: Quality filtering ─────────────────────────────────────────────────
# Same logic as Phase 1 but now applied to ~10M pairs.
# CHANGE: num_proc raised to 8 to use all available CPU cores on the GPU server.
# Lower it back to 4 if you get memory errors.
def is_good_pair(example):
    src = example['src'].strip()
    tgt = example['tgt'].strip()

    if not src or not tgt:           # remove empty pairs
        return False

    src_len = len(src.split())
    tgt_len = len(tgt.split())

    # CHANGE: upper bound raised to 60/70 (was 50/60) to keep more data
    # at full-dataset scale; lower bound stays at 3 to remove noise.
    if src_len < 3 or src_len > 60:
        return False
    if tgt_len < 3 or tgt_len > 70:
        return False

    ratio = src_len / tgt_len
    if ratio < 0.4 or ratio > 2.5:  # same length-ratio filter
        return False

    devanagari = re.compile(r'[\u0900-\u097F]')
    if not devanagari.search(tgt):   # Hindi must have Devanagari
        return False
    if devanagari.search(src):       # English must NOT have Devanagari
        return False

    return True

print(f"Total pairs before filtering: {len(dataset['train']):,}")
print("Filtering... this will take 5-10 minutes on the full dataset.")

filtered = dataset['train'].filter(
    is_good_pair,
    num_proc=8        # CHANGE: 8 cores for the GPU server (was 4)
)
print(f"After filtering:              {len(filtered):,}")
print(f"Kept {len(filtered)/len(dataset['train'])*100:.1f}% of original data")

In [ ]:
# ── Cell 4: Train / Val / Test split ─────────────────────────────────────────
# CHANGE: Phase 1 used a fixed 120K/15K/15K split of 150K.
# Phase 2 uses an 80/10/10 ratio on whatever filtered size we end up with,
# so the split automatically scales with the dataset size.
total      = len(filtered)
train_end  = int(total * 0.80)
val_end    = int(total * 0.90)

# CHANGE: Using shuffle with a fixed seed before selecting, so the split is
# reproducible even if the filter changes slightly between runs.
filtered = filtered.shuffle(seed=42)

train_data = filtered.select(range(0,         train_end))
val_data   = filtered.select(range(train_end, val_end))
test_data  = filtered.select(range(val_end,   total))

print(f"Train : {len(train_data):,}")
print(f"Val   : {len(val_data):,}")
print(f"Test  : {len(test_data):,}")

In [ ]:
# ── Cell 5: Save raw splits to CSV ───────────────────────────────────────────
# CHANGE: Phase 1 saved a single CSV per split (~120K rows fit in memory).
# Phase 2 saves in CHUNKS of 1M rows so we never materialise the full
# DataFrame at once. Each chunk is later read sequentially during preprocessing.
# We also save a small 'raw' CSV for quick inspection.

CHUNK_SIZE = 1_000_000   # 1M rows per chunk file

def save_in_chunks(hf_split, split_name, chunk_size=CHUNK_SIZE):
    """Converts a HuggingFace Dataset split to chunked CSVs."""
    n      = len(hf_split)
    chunks = math.ceil(n / chunk_size)
    print(f"Saving '{split_name}' ({n:,} rows) as {chunks} chunk(s)...")
    for i in range(chunks):
        start = i * chunk_size
        end   = min(start + chunk_size, n)
        chunk = hf_split.select(range(start, end))
        df    = pd.DataFrame({'english': chunk['src'], 'hindi': chunk['tgt']})
        path  = os.path.join(OUTPUT_DIR, f"{split_name}_raw_chunk{i}.csv")
        df.to_csv(path, index=False, encoding='utf-8')
        print(f"  Chunk {i}: rows {start:,}–{end:,} → {path}")
        del df, chunk
        gc.collect()

save_in_chunks(train_data, 'train')
save_in_chunks(val_data,   'val')
save_in_chunks(test_data,  'test')
print("\nAll raw chunks saved!")

In [ ]:
# ── Cell 6: Cleaning functions (same as Phase 1) ─────────────────────────────
def clean_english(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\'\"]-", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_hindi(text):
    text = str(text)
    text = re.sub(r'[^\u0900-\u097F\s\.\,\!\?।]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning functions defined.")

# Quick sanity check
print("EN:", clean_english("Hello, World! 123"))
print("HI:", clean_hindi("नमस्ते दुनिया! Hello"))

In [ ]:
# ── Cell 7: Tokenisation functions ───────────────────────────────────────────
from indicnlp.tokenize import indic_tokenize

def tokenize_english(text):
    return nltk.word_tokenize(text)

def tokenize_hindi(text):
    return indic_tokenize.trivial_tokenize(text, lang='hi')

def add_special_tokens(tokens):
    return ['<SOS>'] + tokens + ['<EOS>']

# Quick sanity check
print(tokenize_english("hello world"))
print(tokenize_hindi("नमस्ते दुनिया"))

In [ ]:
# ── Cell 8: Build vocabulary from TRAINING data only ─────────────────────────
# CHANGE: Phase 1 built vocab by loading the entire train DF into RAM.
# Phase 2 streams through the train CSV chunks one at a time so we never
# need more than ~1M rows in memory.
# CHANGE: min_freq raised from 5 → 10 because with 8M+ training sentences
# there will be millions of unique tokens; keeping only high-freq tokens
# keeps the vocab at a manageable size (~100K–200K) for the embedding matrix.

MIN_FREQ = 10   # CHANGE: was 5 in Phase 1

eng_counter = Counter()
hi_counter  = Counter()

# Find all train chunk files
train_chunks = sorted([
    os.path.join(OUTPUT_DIR, f)
    for f in os.listdir(OUTPUT_DIR)
    if f.startswith('train_raw_chunk') and f.endswith('.csv')
])
print(f"Found {len(train_chunks)} train chunk(s). Building vocab...")

for chunk_path in train_chunks:
    df = pd.read_csv(chunk_path)
    df['english'] = df['english'].apply(clean_english)
    df['hindi']   = df['hindi'].apply(clean_hindi)
    for tokens in df['english'].apply(tokenize_english):
        eng_counter.update(tokens)
    for tokens in df['hindi'].apply(tokenize_hindi):
        hi_counter.update(tokens)
    del df
    gc.collect()
    print(f"  Processed {chunk_path}")

def build_vocab_from_counter(counter, min_freq):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

eng_vocab = build_vocab_from_counter(eng_counter, MIN_FREQ)
hi_vocab  = build_vocab_from_counter(hi_counter,  MIN_FREQ)

print(f"\nEnglish vocabulary size: {len(eng_vocab):,}")
print(f"Hindi vocabulary size:   {len(hi_vocab):,}")

In [ ]:
# ── Cell 9: Save vocabularies ─────────────────────────────────────────────────
with open(os.path.join(OUTPUT_DIR, 'eng_vocab.pkl'), 'wb') as f:
    pickle.dump(eng_vocab, f)
with open(os.path.join(OUTPUT_DIR, 'hi_vocab.pkl'), 'wb') as f:
    pickle.dump(hi_vocab, f)
print("Vocabularies saved!")

In [ ]:
# ── Cell 10: Build FastText embedding matrices ────────────────────────────────
# Same as Phase 1. Download cc.en.300.bin and cc.hi.300.bin from:
#   https://fasttext.cc/docs/en/crawl-vectors.html
# and place them in the working directory (or update the paths below).
import fasttext

print("Loading FastText models... (takes 1-2 minutes each)")
ft_en = fasttext.load_model('cc.en.300.bin')
ft_hi = fasttext.load_model('cc.hi.300.bin')
print("Loaded!")

# CHANGE: vocab sizes are larger now, so matrices will be bigger.
# English: ~100K words × 300 dims ≈ 120 MB
# Hindi:   ~150K words × 300 dims ≈ 180 MB  (Hindi is morphologically richer)
eng_matrix = np.zeros((len(eng_vocab), 300), dtype=np.float32)
for word, idx in tqdm(eng_vocab.items(), desc="English matrix"):
    eng_matrix[idx] = ft_en.get_word_vector(word)

hi_matrix = np.zeros((len(hi_vocab), 300), dtype=np.float32)
for word, idx in tqdm(hi_vocab.items(), desc="Hindi matrix"):
    hi_matrix[idx] = ft_hi.get_word_vector(word)

np.save(os.path.join(OUTPUT_DIR, 'eng_embedding_matrix.npy'), eng_matrix)
np.save(os.path.join(OUTPUT_DIR, 'hi_embedding_matrix.npy'),  hi_matrix)
print(f"English matrix: {eng_matrix.shape}  {eng_matrix.nbytes/1024/1024:.1f} MB")
print(f"Hindi matrix:   {hi_matrix.shape}    {hi_matrix.nbytes/1024/1024:.1f} MB")
print("Embedding matrices saved!")

# Free FastText models — they use a lot of RAM
del ft_en, ft_hi
gc.collect()

In [ ]:
# ── Cell 11: Tokenise, index, pad and save — chunk by chunk ──────────────────
# CHANGE: Phase 1 loaded all data into RAM, tokenised everything, then saved
# one big .pkl per split.  That works for 150K rows but not for 8M+.
# Phase 2 processes one CSV chunk at a time, appends the result to a
# *processed* CSV chunk (with eng_padded / hi_padded columns stored as
# space-separated strings), then frees the memory before moving on.
# The model DataLoader will read these chunks and reconstruct tensors.

MAX_LEN = 60   # CHANGE: was 50 in Phase 1; raised to 60 to handle longer sentences

def tokens_to_indices(tokens, vocab):
    return [vocab.get(t, vocab['<UNK>']) for t in tokens]

def pad_sequence(indices, max_len):
    if len(indices) >= max_len:
        return indices[:max_len]
    return indices + [0] * (max_len - len(indices))

def process_chunk(df):
    """Clean → tokenise → add specials → index → pad a single DataFrame chunk."""
    df = df.copy()
    df['english'] = df['english'].apply(clean_english)
    df['hindi']   = df['hindi'].apply(clean_hindi)

    df['eng_tokens'] = df['english'].apply(lambda t: add_special_tokens(tokenize_english(t)))
    df['hi_tokens']  = df['hindi'].apply(lambda t: add_special_tokens(tokenize_hindi(t)))

    df['eng_indices'] = df['eng_tokens'].apply(lambda t: tokens_to_indices(t, eng_vocab))
    df['hi_indices']  = df['hi_tokens'].apply(lambda t: tokens_to_indices(t, hi_vocab))

    # Store padded sequences as space-separated strings so they survive CSV round-trips
    df['eng_padded'] = df['eng_indices'].apply(
        lambda x: ' '.join(map(str, pad_sequence(x, MAX_LEN)))
    )
    df['hi_padded']  = df['hi_indices'].apply(
        lambda x: ' '.join(map(str, pad_sequence(x, MAX_LEN)))
    )

    # Keep only the columns the model actually needs
    return df[['english', 'hindi', 'eng_padded', 'hi_padded']]


for split_name in ['train', 'val', 'test']:
    raw_chunks = sorted([
        os.path.join(OUTPUT_DIR, f)
        for f in os.listdir(OUTPUT_DIR)
        if f.startswith(f'{split_name}_raw_chunk') and f.endswith('.csv')
    ])
    print(f"\nProcessing '{split_name}' ({len(raw_chunks)} chunk(s))...")
    for i, chunk_path in enumerate(tqdm(raw_chunks, desc=split_name)):
        df_raw  = pd.read_csv(chunk_path)
        df_proc = process_chunk(df_raw)
        out_path = os.path.join(OUTPUT_DIR, f"{split_name}_processed_chunk{i}.csv")
        df_proc.to_csv(out_path, index=False, encoding='utf-8')
        del df_raw, df_proc
        gc.collect()
    print(f"  '{split_name}' done.")

print("\nAll splits processed and saved!")

In [ ]:
# ── Cell 12: Save MAX_LEN so model notebooks can read it ─────────────────────
config = {'MAX_LEN': MAX_LEN, 'MIN_FREQ': MIN_FREQ}
with open(os.path.join(OUTPUT_DIR, 'preprocessing_config.pkl'), 'wb') as f:
    pickle.dump(config, f)
print("Config saved:", config)

In [ ]:
# ── Cell 13: Sanity check — read back and inspect one example ────────────────
sample = pd.read_csv(os.path.join(OUTPUT_DIR, 'train_processed_chunk0.csv'), nrows=3)
for _, row in sample.iterrows():
    print("English sentence : ", row['english'])
    print("Hindi sentence   : ", row['hindi'])
    print("eng_padded (ids) : ", row['eng_padded'][:60], "...")
    print("hi_padded  (ids) : ", row['hi_padded'][:60],  "...")
    print("-" * 60)

print(f"\nVocab sizes  → English: {len(eng_vocab):,}  |  Hindi: {len(hi_vocab):,}")
print(f"MAX_LEN      → {MAX_LEN}")
print(f"Embedding    → English: {eng_matrix.shape}  |  Hindi: {hi_matrix.shape}")

In [ ]:
# ── Cell 14: List all output files ───────────────────────────────────────────
print("Files saved in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f:<50}  {size_mb:>8.1f} MB")